In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import max_error
import pandas as pd
import os
import joblib
from itertools import compress

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes.data import besInferenceDatapoints
from neuro_bes.preprocessing import profile_transform
from neuro_bes.postprocessing import evaluation

In [ ]:
path="/home/molnarbalazs/data/BES_ML_modelling/W7X_op23_SPADE_recons/test_newcuration"
file_list=os.listdir(path)
file_list=[i for i in file_list if "_we" in i]
batch_test=[]
for idx,db_file in enumerate(file_list):
    bes_data=besInferenceDatapoints(path=os.path.join(path,db_file))
    mask=np.max(bes_data.emissions[:,-10:],axis=1)==np.max(bes_data.emissions,axis=1)
    #filter out nan profiles
    mask=np.isnan(bes_data.emissions).any(axis=1) | mask
    mask=np.mean(bes_data.emissions,axis=1)<30 | mask
    bes_data.emissions=bes_data.emissions[~mask]
    bes_data.densities=bes_data.densities[~mask]
    bes_data.tags=list(compress(bes_data.tags,~mask))
    if bes_data.emissions.shape[0]>1:
        if "slow modulation" in bes_data.verbose:
            batch_test.append(bes_data)

In [ ]:
plt.plot(batch_test[0].emissions.T)

In [ ]:
pipeline=joblib.load("preprocessing_pipeline_newcuration_v2.joblib")
model=tf.keras.models.load_model("density_prediction_model_newcuration_v2.keras")
pipeline_batch_test = pipeline.transform(batch_test)

In [ ]:
for batch in pipeline_batch_test:
    y_pred_scaled = model.predict(batch.emissions)
    y_pred_scaled=y_pred_scaled.reshape(y_pred_scaled.shape[0], -1)
    batch.densities=y_pred_scaled

In [ ]:
batch_test_pred=pipeline.inverse_transform(pipeline_batch_test)
interpolator=profile_transform.InterpolateToCommonGrid()
batch_test_pred=interpolator.fit_transform(batch_test_pred)
batch_test_on_common_grid=interpolator.transform(batch_test)
y_pred=np.concatenate([batch.densities for batch in batch_test_pred])
y_true=np.concatenate([batch.densities for batch in batch_test_on_common_grid])
x_test=np.concatenate([batch.emissions for batch in batch_test_on_common_grid])
r_coord=interpolator.common_grid_

In [ ]:
#bes_data_to_plot=batch_test_pred[38]
#bes_data_to_plot=batch_test_pred[29]
#bes_data_to_plot=batch_test_pred[45]
bes_data_to_plot_pred=batch_test_pred[10]
bes_data_to_plot_true=batch_test_on_common_grid[10]
print(bes_data_to_plot_pred.ID)
plt.figure(figsize=(20,5))
colors = plt.cm.viridis(np.linspace(0, 1, bes_data_to_plot_pred.densities.shape[0]))
ymax=max(np.max(bes_data_to_plot_pred.densities), np.max(bes_data_to_plot_true.densities))*1.1
plt.subplot(1,4,1)
for i in range(bes_data_to_plot_true.emissions.shape[0]):
    plt.plot(bes_data_to_plot_true.grid, bes_data_to_plot_true.emissions[i], color=colors[i])
plt.subplot(1,4,2)
for i in range(bes_data_to_plot_pred.densities.shape[0]):
    plt.plot(bes_data_to_plot_pred.grid, bes_data_to_plot_pred.densities[i], color=colors[i])
    plt.ylim(0,ymax)
plt.subplot(1,4,3)
for i in range(bes_data_to_plot_true.densities.shape[0]):
    plt.plot(bes_data_to_plot_true.grid, bes_data_to_plot_true.densities[i], color=colors[i])
    plt.ylim(0,ymax)
plt.subplot(1,4,4)
for i in range(bes_data_to_plot_pred.densities.shape[0]):
    err = np.abs(bes_data_to_plot_pred.densities[i] - bes_data_to_plot_true.densities[i])
    plt.plot(bes_data_to_plot_true.grid, err, color=colors[i])
    plt.ylim(0,ymax)
#plt.legend(["profile "+str(i) for i in range(bes_data_to_plot.densities.shape[0])])



In [ ]:
plt.plot(bes_data_to_plot_true.emissions[60:65].T)

In [ ]:
time_instances=[float(tag[14:18]) for tag in bes_data_to_plot_true.tags]
light_max_location=r_coord[np.argmax(bes_data_to_plot_true.emissions,axis=1)]

In [ ]:
# plot the density and prediction profiles on a heatmap with same color scale
# set seismic colormap with white at 0 and red/blue for positive/negative values

plt.figure(figsize=(10,12))
plt.subplot(4,1,1)
# colorbar with same scale for both plots
im1 = plt.pcolormesh(time_instances, r_coord, bes_data_to_plot_true.emissions.T, cmap='RdBu_r')
#add light maximum location as black line
plt.plot(time_instances, light_max_location, color='k', label='Light Max Location',linewidth=.5)
plt.legend(fontsize=12)
cbar = plt.colorbar(im1)
cbar.ax.set_ylabel('mV',fontsize=18, labelpad=10)
cbar.ax.tick_params(labelsize=12)
plt.title('Light Profiles', fontsize=20)
plt.xlabel('Time (s)',fontsize=18, labelpad=10)
plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.subplot(4,1,2)
# colorbar with same scale for both plots
im2 = plt.pcolormesh(time_instances, r_coord, bes_data_to_plot_true.densities.T*1e19, cmap='RdBu_r')
#add light maximum location as black line
plt.plot(time_instances, light_max_location, color='k', label='Light Max Location',linewidth=.5)
plt.legend(fontsize=12)
cbar = plt.colorbar(im2)
cbar.ax.set_ylabel('$m^{-3}$',fontsize=18, labelpad=10)
cbar.ax.tick_params(labelsize=12)
vmax = max(np.max(bes_data_to_plot_pred.densities*1e19), np.max(bes_data_to_plot_true.densities*1e19))
im2.set_clim(0, vmax)
plt.title('True Density Profiles', fontsize=20)
plt.xlabel('Time (s)',fontsize=18, labelpad=10)
plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.subplot(4,1,3)
im3 = plt.pcolormesh(time_instances,r_coord, bes_data_to_plot_pred.densities.T*1e19, cmap='RdBu_r')
plt.plot(time_instances, light_max_location, color='k', label='Light Max Location',linewidth=.5)
plt.legend(fontsize=12)
cbar = plt.colorbar(im3)
cbar.ax.set_ylabel('$m^{-3}$',fontsize=18, labelpad=10)
cbar.ax.tick_params(labelsize=12)
vmax = max(np.max(bes_data_to_plot_pred.densities*1e19), np.max(bes_data_to_plot_true.densities*1e19))
im3.set_clim(0, vmax)
plt.title('Predicted Density Profiles', fontsize=20)
plt.xlabel('Time (s)',fontsize=18, labelpad=10)
plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
# show difference between density and prediction
plt.subplot(4,1,4)
im4=plt.pcolormesh(time_instances, r_coord, bes_data_to_plot_pred.densities.T*1e19 - bes_data_to_plot_true.densities.T*1e19, cmap='RdBu_r')
plt.plot(time_instances, light_max_location, color='k', label='Light Max Location',linewidth=.5)
plt.legend(fontsize=12)
cbar = plt.colorbar(im4)
cbar.ax.set_ylabel('$m^{-3}$',fontsize=18, labelpad=10)
cbar.ax.tick_params(labelsize=12)
vmax = max(np.max(bes_data_to_plot_pred.densities*1e19), np.max(bes_data_to_plot_true.densities*1e19))
im4.set_clim(-vmax/4, vmax/4)
plt.title('Absolute error', fontsize=20)
plt.xlabel('Time (s)',fontsize=18, labelpad=10)
plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
#plt.savefig("time_evolution_shot_"+bes_data_to_plot.ID+".png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
channel=17
fig, ax = plt.subplots(figsize=(10, 6))
plt.plot(time_instances, bes_data_to_plot_true.densities[:,channel]*1e19, color="grey", linewidth=3, linestyle='dashed', label="True Density")
plt.plot(time_instances, bes_data_to_plot_pred.densities[:,channel]*1e19, color="midnightblue", linewidth=4, label="Predicted Density")
ax.set_title('Electron density at '+str(r_coord[channel])[:4]+' m radius', fontsize=26, pad=15)
ax.set_xlabel('Time (s)', fontsize=18, labelpad=10)
ax.set_ylabel('Electron density ($m^{-3}$)', fontsize=18, labelpad=10)

# Increase tick label size
ax.tick_params(axis='both', which='major', labelsize=18)

# Grid and legend
ax.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=20)
# Tight layout for spacing
plt.tight_layout()
#plt.savefig("channel_"+str(channel)+"_"+bes_data_to_plot.ID+".png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
path="/home/molnarbalazs/data/BES_ML_modelling/W7X_op23_SPADE_recons/train_newcuration"
file_list=os.listdir(path)
file_list=[i for i in file_list if "_we" in i]
batch_train=[]
for idx,db_file in enumerate(file_list):
    bes_data=besInferenceDatapoints(path=os.path.join(path,db_file))
    if bes_data.emissions.shape[0]>1:
        batch_train.append(bes_data)
batch_train=interpolator.transform(batch_train)
x_train=np.concatenate([batch.emissions for batch in batch_train])
y_train=np.concatenate([batch.densities for batch in batch_train])
#y_naive=np.mean(y_train, axis=0)

x_train_sq = np.sum(x_train**2, axis=1)
x_test_sq = np.sum(x_test**2, axis=1)
dist_sq = x_test_sq[:, None] + x_train_sq[None, :] - 2 * x_test.dot(x_train.T)
best_idx = np.argmin(dist_sq, axis=1)
y_naive = y_train[best_idx]


In [ ]:
metrics={'loss_function':'mse'}
evaluation.calc_mape_stats(y_true,y_pred,metrics)
#evaluation.calc_mase_stats(y_true,y_pred,Y_train,metrics)
evaluation.calc_mre(y_true,y_pred,metrics)
#metrics['loss']=model.history.history['loss']
#metrics['val_loss']=model.history.history['val_loss']

In [ ]:
y_true.shape

In [ ]:
plt.plot(r_coord,metrics['MRE'])


In [ ]:
#np.save('mre_slowmod.npy', metrics['MRE']) 

In [ ]:
mase=np.mean(np.abs(y_true-y_pred),axis=1)/np.mean(np.abs((y_true-y_naive)),axis=1)

In [ ]:
plt.hist(mase,bins=100, log =False)
#np.save('mase.npy', mase)

In [ ]:
y_pred_worst=y_pred[mase>1]
y_true_worst=y_true[mase>1]
x_test_worst=x_test[mase>1]
y_pred_best=y_pred[mase<1]
y_true_best=y_true[mase<1]
x_test_best=x_test[mase<1]

In [ ]:
y_pred_best.shape, y_pred_worst.shape

In [ ]:
plt.subplot(2,2,1)
plt.plot(r_coord,y_true_best.T)
plt.subplot(2,2,3)
plt.plot(r_coord,x_test_best.T)
plt.subplot(2,2,2)
plt.plot(r_coord,y_true_worst.T)
plt.subplot(2,2,4)
plt.plot(r_coord,x_test_worst.T)

In [ ]:
whichprofile=200
plt.plot(r_coord,y_true[whichprofile,:])
plt.plot(r_coord,y_pred[whichprofile,:])
plt.plot(r_coord,y_naive[whichprofile,:])
#np.save('good_mase_y_pred.npy',y_pred[whichprofile,:])
#np.save('good_mase_y_true.npy',y_true[whichprofile,:])
#np.save('good_mase_y_naive.npy',y_naive[whichprofile,:])

In [ ]:
whichprofile=1053
plt.plot(r_coord,y_true[whichprofile,:])
plt.plot(r_coord,y_pred[whichprofile,:])
plt.plot(r_coord,y_naive[whichprofile,:])
#np.save('bad_mase_y_pred.npy',y_pred[whichprofile,:])
#np.save('bad_mase_y_true.npy',y_true[whichprofile,:])
#np.save('bad_mase_y_naive.npy',y_naive[whichprofile,:])

In [ ]:
best_mase=np.argmin(mase)
plt.plot(r_coord,y_pred[best_mase,:])
plt.plot(r_coord,y_true[best_mase,:])
#np.save('best_mase_y_pred.npy',y_pred[best_mase,:])
#np.save('best_mase_y_true.npy',y_true[best_mase,:])
#np.save('rcoord.npy',r_coord)

In [ ]:
best_l2=np.argmin(np.mean((y_pred-y_true)**2,axis=1))
plt.subplot(1,2,1)
plt.plot(r_coord,y_pred[best_l2,:])
plt.plot(r_coord,y_true[best_l2,:])
plt.subplot(1,2,2)
plt.plot(r_coord,x_test[best_l2,:])

In [ ]:
worst_mase=np.argmax(mase)
plt.subplot(1,2,1)
plt.plot(r_coord,y_pred[worst_mase,:])
plt.plot(r_coord,y_true[worst_mase,:])
#np.save('worst_mase_y_pred.npy',y_pred[worst_mase,:])
#np.save('worst_mase_y_true.npy',y_true[worst_mase,:])
plt.subplot(1,2,2)
plt.plot(r_coord,x_test[worst_mase,:])

In [ ]:
worst_l2=np.argmax(np.mean((y_pred-y_true)**2,axis=1))
plt.subplot(1,2,1)
plt.plot(r_coord,y_pred[worst_l2,:])
plt.plot(r_coord,y_true[worst_l2,:])
plt.subplot(1,2,2)
plt.plot(r_coord,x_test[worst_l2,:])

In [ ]:
worst_l1=np.argmax(np.mean(np.abs(y_pred-y_true),axis=1))
plt.subplot(1,2,1)
plt.plot(r_coord,y_pred[worst_l1,:])
plt.plot(r_coord,y_true[worst_l1,:])
plt.subplot(1,2,2)
plt.plot(r_coord,x_test[worst_l1,:])

In [ ]:
maximum=max_error(y_pred[0,:],y_true[0,:])
worst_pointwise=0
for i in range(y_pred.shape[0]):
    m=max_error(y_pred[i,:],y_true[i,:])
    if maximum<m:
        maximum=m
        worst_pointwise=i
plt.subplot(1,2,1)
plt.plot(r_coord,y_pred[worst_pointwise,:])
plt.plot(r_coord,y_true[worst_pointwise,:])
plt.subplot(1,2,2)
plt.plot(r_coord,x_test[worst_pointwise,:])

In [ ]:
def MC_dropout(model, x_test, x_scaler, y_test, y_scaler, MC_dropout_samples=100):
    r = np.arange(len(y_test))
    plt.plot(r,y_test)
    pred_mean=np.zeros(len(y_test))
    pred_sq=np.zeros(len(y_test))
    for i in range(MC_dropout_samples):
        pred=np.squeeze(y_scaler.inverse_transform(model(x_scaler.transform(x_test).reshape(1, -1),training=True)[0]))
        pred_mean=pred_mean+pred/MC_dropout_samples
        pred_sq=pred_sq+pred**2/MC_dropout_samples
    pred_std=np.sqrt(pred_sq-pred_mean**2)
    plt.plot(r,pred_mean)
    plt.fill_between(r,pred_mean-pred_std,pred_mean+pred_std,color="orange",alpha=0.3)
    plt.show()

In [ ]:
#MC_dropout(model, X_test[worst_l2,:], x_scaler, Y_test[worst_l2,:], y_scaler, MC_dropout_samples=100)

In [ ]:
def max_l2_difference(X: np.ndarray) -> float:
    # Compute the squared norms of each row
    norms_squared = np.sum(X ** 2, axis=1, keepdims=True)  # shape (n, 1)

    # Compute pairwise squared L2 distances using the identity:
    # ||a - b||^2 = ||a||^2 + ||b||^2 - 2·a·b
    dists_squared = norms_squared + norms_squared.T - 2 * X @ X.T

    return np.unravel_index(np.argmax(dists_squared),dists_squared.shape)

In [ ]:
ind1,ind2=max_l2_difference(y_true)
plt.plot(r_coord,y_true[ind1,:])
plt.plot(r_coord,y_true[ind2,:])

In [ ]:
plt.plot(r_coord,x_test[ind1,:])
plt.plot(r_coord,x_test[ind2,:])

In [ ]:
x_test_sample=batch_test_pred[1].emissions[17]/300
y_pred_sample=model.predict(x_test_sample.reshape(1, -1))
plt.subplot(1,2,1)
plt.plot(r_coord,x_test_sample)
plt.subplot(1,2,2)
plt.plot(r_coord,y_pred_sample[0,:])
plt.plot(r_coord,y_pred[25,:])
plt.plot(r_coord,y_true[25,:])

In [ ]:
plt.plot(r_coord,y_pred[ind1,:])
plt.plot(r_coord,y_true[ind1,:])

In [ ]:
plt.plot(r_coord,y_pred[ind2,:])
plt.plot(r_coord,y_true[ind2,:])

In [ ]:
df = pd.Series(metrics)
print(df)
#df.to_csv('out_temp.csv')